# Training Notebook

### Objectives:
- Create a random forest regressor based model
- Use scikit-learn
- Explore the data to find the most important deciders of weather the flight is delayed
- Graph these explorations
- Split the datasets
- Get decent accuracy with the validation dataset

In [15]:
# Scikit imports
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [16]:
# Imports
import duckdb as ddb
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from datetime import datetime as dt

# Constants
BACKEND_ROOT = Path().cwd().resolve().parents[2]
DUCKDB_PATH = BACKEND_ROOT/"data/duck_database.duckdb"
FIGURES_PATH = BACKEND_ROOT/"src/ml/training/figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
TRAIN_RESULTS_PATH = FIGURES_PATH/"train_results"
TRAIN_RESULTS_PATH.mkdir(parents=True, exist_ok=True)
SIGNIFICANT_DELAY_MINUTES = 25

In [17]:
# Getting data_df
con = ddb.connect(DUCKDB_PATH)
raw_data_df = con.sql("""
    SELECT * FROM model_dataset LIMIT 500
""").df()
con.close()

data_df = raw_data_df.dropna(axis=0)

In [18]:
# Create X's and y's
x_numeric_features = ['year', 'month', 'day_of_month', 'day_of_week',
        'pred_dep_time', 'pred_arr_time', 'pred_elapsed_time',
       'fl_distance', 'origin_weather_code',
       'origin_temperature_2m_max', 'origin_temperature_2m_min',
       'origin_apparent_temperature_max', 'origin_apparent_temperature_min',
       'origin_precipitation_sum', 'origin_rain_sum', 'origin_showers_sum',
       'origin_snowfall_sum', 'origin_cloud_cover_mean',
       'origin_wind_speed_10m_max', 'origin_wind_gusts_10m_max',
       'origin_wind_direction_10m_dominant', 'origin_pressure_msl_mean',
       'dest_weather_code', 'dest_temperature_2m_max',
       'dest_temperature_2m_min', 'dest_apparent_temperature_max',
       'dest_apparent_temperature_min', 'dest_precipitation_sum',
       'dest_rain_sum', 'dest_showers_sum', 'dest_snowfall_sum',
       'dest_cloud_cover_mean', 'dest_wind_speed_10m_max',
       'dest_wind_gusts_10m_max', 'dest_wind_direction_10m_dominant',
       'dest_pressure_msl_mean']
x_categorical_features = ['flight_date', 'origin', 'dest']
x_features = x_numeric_features + x_categorical_features

X = data_df[x_features]
y = (data_df["delay"] >= SIGNIFICANT_DELAY_MINUTES).astype(int)

print(f"Significant delay threshold: {SIGNIFICANT_DELAY_MINUTES} minutes")
print(y.value_counts(normalize=True).rename({0: "not_significant", 1: "significant"}))

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1,
    stratify=y,
)


Significant delay threshold: 25 minutes
delay
not_significant    0.96
significant        0.04
Name: proportion, dtype: float64


### Data exploration
- With notes on matplotlab (not great with it)

In [19]:
# Correlations

numeric_features = data_df[x_numeric_features].copy()
numeric_features["significant_delay"] = y.to_numpy()
target_correlations = (
    numeric_features
    .corr(numeric_only=True)["significant_delay"]
    .drop("significant_delay")
    .dropna()
    .sort_values(key=lambda values: values.abs())
)

fig, ax = plt.subplots(figsize=(11, max(8, 0.34 * len(target_correlations))))
colors = ["#b45309" if value < 0 else "#0f766e" for value in target_correlations]

ax.barh(target_correlations.index, target_correlations.values, color=colors)
ax.axvline(0, color="#222222", linewidth=0.8)
ax.set_title("Pearson correlation with significant delay target")
ax.set_xlabel(f"Correlation with significant delay (>={SIGNIFICANT_DELAY_MINUTES} mins)")
ax.set_ylabel("Numeric feature")
ax.grid(axis="x", alpha=0.25)

fig.tight_layout()
fig.savefig(FIGURES_PATH/ "correlation_chart.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [20]:
# Distribution
fig, ax = plt.subplots(figsize=(10, 6))

target_distribution = y.value_counts().sort_index().rename({0: "not_significant", 1: "significant"})
bar_colors = ["#0f766e", "#b45309"]

ax.bar(target_distribution.index, target_distribution.values, color=bar_colors, edgecolor="white")

ax.set_title("Significant delay target distribution")
ax.set_xlabel(f"Target class (>={SIGNIFICANT_DELAY_MINUTES} mins)")
ax.set_ylabel("Number of flights")
ax.grid(axis="y", alpha=0.25)
ax.bar_label(ax.containers[0], fmt="%d")

fig.tight_layout()
fig.savefig(FIGURES_PATH/ "delay_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [22]:
# Missing values

missing_source = raw_data_df.loc[raw_data_df["delay"].notna(), x_features + ["delay"]].copy()
missing_source["significant_delay"] = (
    missing_source["delay"] >= SIGNIFICANT_DELAY_MINUTES
).astype(int)
missing_by_target = (
    missing_source
    .groupby("significant_delay")[x_features]
    .apply(lambda frame: frame.isna().mean())
    .T
    .rename(columns={0: "not_significant", 1: "significant"})
    .sort_values(by=["significant", "not_significant"], ascending=False)
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, max(8, 0.34 * len(missing_by_target))),
    sharey=True,
)

for ax, label, color in zip(
    axes,
    ["not_significant", "significant"],
    ["#0f766e", "#b45309"],
):
    ax.barh(missing_by_target.index, missing_by_target[label], color=color)
    ax.set_title(label.replace("_", " ").title())
    ax.set_xlabel("Fraction missing")
    ax.grid(axis="x", alpha=0.25)

axes[0].set_ylabel("Feature")
fig.suptitle(f"Missing values by significant delay target (>={SIGNIFICANT_DELAY_MINUTES} mins)")

fig.tight_layout()
fig.savefig(FIGURES_PATH/ "missing_values.png", dpi=300, bbox_inches="tight")
plt.close(fig)

### Training
Methods for accuracy improvments:
- Duplicate accurate indicators (shown in correlation img)
- Combine features, for example: snow + wind
- Train many models

In [ ]:
# DummyClassifier Baseline
dummy_strats = ['most_frequent', 'stratified', 'uniform']

test_scores = []
for s in dummy_strats:
    dclf = DummyClassifier(strategy = s, random_state = 0)
        
    dclf.fit(X_train, y_train)
    score = dclf.score(X_test, y_test)
    test_scores.append(score)

# dummy results
ax = sns.stripplot(x=dummy_strats, y=test_scores)
ax.set_xlabel("Dummy strategies")
ax.set_ylabel("Results")
plt.savefig(FIGURES_PATH/'dummy_strats.png')
plt.close()


In [15]:
# Random forest classifier search. Keep this modest enough for a short AWS run.
param_grid = {
    "classifier__n_estimators": [150, 250, 350],
    "classifier__max_depth": [12, 18, 24, None],
    "classifier__min_samples_split": [5, 10, 20],
    "classifier__min_samples_leaf": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.5],
    "classifier__bootstrap": [True],
    "classifier__max_samples": [0.7, 0.9],
    "classifier__class_weight": ["balanced", "balanced_subsample"],
}

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), x_categorical_features),
        ("numeric", "passthrough", x_numeric_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", RandomForestClassifier(n_jobs=1, random_state=1)),
    ]
)

grid_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=24,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=1,
)

grid_search.fit(X_train, y_train)


NameError: name 'X_train' is not defined

In [5]:
preds = grid_search.predict(X_test)
pred_probs = grid_search.predict_proba(X_test)[:, 1]

results = f"""
Run: {dt.now().strftime("%Y-%m-%d -- %H-%M-%S")}
Best CV F1: {grid_search.best_score_:.4f}
Accuracy: {accuracy_score(y_test, preds):.4f}
Precision: {precision_score(y_test, preds, zero_division=0):.4f}
Recall: {recall_score(y_test, preds, zero_division=0):.4f}
F1: {f1_score(y_test, preds, zero_division=0):.4f}
ROC AUC: {roc_auc_score(y_test, pred_probs):.4f}
=== space ===
=== space ===
"""
with open(TRAIN_RESULTS_PATH/"best_results.txt", "a") as f:
    try:
        f.write(results)
    except:
        print("Error writing results from train, check terminal")
    finally:
        print(results)

NameError: name 'grid_search' is not defined